# Congestion Rent: Pricing Europe's Grid Bottlenecks

The European Union aims to achieve [at least 42.5% power generation](https://energy.ec.europa.eu/topics/renewable-energy_en) produced by renewables by 2030. Consequently, there is a lot of chatter around what that power generation mix could and should look like, the type and amount of investments required, contracts for differences, etc. In comparison, little discussion is dedicated towards what we believe will be the true bottleneck to the energy transition: *congestion*.

In Europe's [zonal system](https://www.acer.europa.eu/electricity/market-rules/capacity-allocation-and-congestion-management/bidding-zone-review), when two neighbouring bidding zones have different [day-ahead prices](https://www.epexspot.com/en/basicspowermarket#day-ahead-and-intraday-the-backbone-of-the-european-spot-market), power is supposed to flow from the cheap zone to the expensive one until prices converge. In a large portion of operating time, the interconnector between them is already running at its physical capacity limit (its net transfer capacity, or NTC). 

This is called **congestion**. And every hour a border sits congested, that price spread gets a measurable value attached to it: **congestion rent**.

This notebook estimates congestion rent directly from flow data &ndash; for every congested observation, `|price difference| x |flow| x duration` gives the implied congestion rent in EUR &ndash; and then works through:

<!-- TODO - rewrite once structure is finalised -->

- a rolling 30-day leaderboard of Europe's highest-rent borders, plus the daily rent trend behind it
- a case study of 2025's most widely congested moment, ranked by how many borders were congested at once (a different question from which border is most expensive)
- monthly and yearly leaderboards across four competing metrics, since "worst border" depends on which one you ask
- a concentration check &ndash; do a handful of borders really dominate the total? &ndash; and a breakdown by region and country

**Before you run this:** sign up at [voltpath.ai/dashboard](https://voltpath.ai/dashboard) for a free personal API key (copy it any time from [voltpath.ai/settings](https://voltpath.ai/settings)), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY`.

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai)). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.

In [ ]:
# common.py is shared across every article and changes often; without this a
# kernel that already imported it keeps serving the stale module.
%load_ext autoreload
%autoreload 2

import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from common import (
    FLOW_COLORS,
    border_count_price_chart,
    ccr_mechanism_treemap,
    concentration_pie_chart,
    congestion_rent_schematic,
    daily_rent_chart,
    drop_ukraine,
    fetch_flows,
    fetch_flows_range,
    flow_type_summary,
    hourly_border_counts,
    leaderboard_subplots,
    region_treemap,
    save_chart,
    top_n_bar,
    top_vs_rest,
)

## Computing congestion rent

Congestion rent is **not** the same thing as economic cost or lost welfare.

Congestion rent is price gap x volume actually traded, governed under [Regulation (EU) 2019/943, Art. 19](https://eur-lex.europa.eu/eli/reg/2019/943/oj/eng). ENTSO-E's [TSOs' proposal for the Use of Congestion Income &ndash; explanatory document](https://eepublicdownloads.entsoe.eu/clean-documents/cep/200703_Use_of_Congestion_Income_-_Explanatory_Document.pdf) has the theoretical and regulatory grounding that illustrates how congestion rent can be constructed from Net Export and Net Import curves.

The key idea to note is that the congestion rent is simply a rectangle, not the [deadweight-loss triangle](https://en.wikipedia.org/wiki/Deadweight_loss) one would get from integrating under each zone's demand/supply curve (as that requires bid-curve data). See the schematic below.

Congestion rent is a directly observable severity signal. It doesn't tell us how much better off Europe would be without the bottleneck, but it does show how much value the constraint is carrying, which makes it a useful severity signal.

In [ ]:
fig = congestion_rent_schematic()
save_chart(fig, "congestion-rent-vs-dwl-schematic", scale=2)
fig.show()

## Pulling flow data

Before diving into analysis, we need to obtain flow data. VoltPath's `GET /api/v1/flows` already joins physical flow with day-ahead prices on both sides of the border and classifies each observation into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage` &ndash; see [Flow Types](https://docs.voltpath.ai/concepts/flow-types) for the full definitions.

We will pull the last 30 days across every border at native resolution &ndash; see [Response Format: Resolution](https://docs.voltpath.ai/concepts/response-format#resolution) for further information.

In [ ]:
now = datetime.now(UTC)
flows_30d = fetch_flows_range((now - timedelta(days=30)).strftime("%Y-%m-%d"), now.strftime("%Y-%m-%d"))
print(f"{len(flows_30d):,} observations across {flows_30d['border'].nunique()} borders")
flows_30d.head()

To get a sense of the scale of congestion, the total congestion rent for the last 30 days is calculated below.

In [ ]:
congested = flows_30d[flows_30d["flow_type"] == "Congestion"].copy()
congested["rent_eur"] = congested["value_eur"].abs()

total_rent = congested["rent_eur"].sum()
print(f"{len(congested):,} congested observations, congestion rent: EUR {total_rent:,.0f}")

## Case study: 2025's most widely congested moment

To visualise what congestion looks like, let's ask the following question: when was the grid most *widely* constrained? In other words, was there a single moment with the most borders congested at once?

We scan every Market Time Unit (MTU) across 2025, count how many borders are flagged `Congestion`, then take the timestamps with the highest counts and switch to VoltPath's [dashboard](https://voltpath.ai/dashboard/).

In [ ]:
flows_2025 = fetch_flows_range("2025-01-01", "2026-01-01")
counts_2025 = hourly_border_counts(flows_2025, "Congestion")
total_borders = flows_2025["border"].nunique()

max_borders = counts_2025["n_borders"].max()
tied = counts_2025[counts_2025["n_borders"] == max_borders].sort_values("datetime")
print(
    (
        f"Peak: {max_borders} of {total_borders} borders simultaneously congested, "
        f"{len(tied)} timestamp(s) tied at that max:"
    )
)
tied

The earliest of those tied peaks falls on 2025-01-28. The [VoltPath dashboard for that day](https://voltpath.ai/dashboard?viewMode=flows-prices&dateFrom=2025-01-28&dateTo=2025-01-29) &ndash; with the playback control dragged to 11:15 UTC &ndash; shows a swarm of red arrows.

<img src="media/congestion-peak-2025-01-28-dashboard.png" alt="VoltPath dashboard showing peak simultaneous congestion on 2025-01-28" width="800">

The chart below plots that day's congested-border count against the average day-ahead price across all zones, with the peak moment marked. Both the day charted and the marker are derived from the scan above rather than hardcoded.

In [ ]:
peak_moment = tied["datetime"].iloc[0]
flows_peak_day = fetch_flows(
    peak_moment.strftime("%Y-%m-%d"),
    (peak_moment + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    resolution="native",
)

fig = border_count_price_chart(
    flows_peak_day,
    "Congestion",
    f"Congested borders vs. average day-ahead price, {peak_moment:%Y-%m-%d} (UTC)",
    peak_vline=peak_moment.strftime("%Y-%m-%d %H:%M:%S"),
    peak_label="Peak congestion",
)
save_chart(fig, f"congestion-peak-{peak_moment:%Y-%m-%d}")
fig.show()

### Monthly leaderboard

Rather than ranking borders by a single day, let's examine how borders do across a full month. And let's go beyond counting borders.

In terms of metrics, we came up with 4 options:

- `n_hours` and `volume_mwh` answer "how often" and "how much energy"
- `implied_value_eur` answers "how much value the price gap is capturing on this border" (under implicit auctions, roughly the congestion income TSOs collect from market participants)
- `avg_value_per_hour_eur` is an *intensity* metric (rare-but-severe vs. mild-but-frequent), not a welfare number &ndash; useful for spotting outlier hours, not for sizing total impact

All four panels come from `leaderboard_subplots` in `common.py`, and honour the `exclude_ukraine` flag set above.

Set `selected_month` below (e.g. `"2023-02"`) to pick a month; leave it `None` to default to the last full calendar month.

Ukraine's `UA_IPS` borders (`HU/PL/RO/SK -> UA_IPS`) are a special case since 2022: wartime grid conditions and emergency EU synchronisation, not ordinary market congestion, drive their numbers, so they can dominate a leaderboard for reasons unrelated to everything else on it. The `exclude_ukraine` flag below drops them from every leaderboard in this notebook.

In [ ]:
selected_month = None  # e.g. "2023-02"; None = last full calendar month
exclude_ukraine = True  # special case due to the ongoing war

month = (
    pd.Period(selected_month, freq="M")
    if selected_month
    else pd.Period(datetime.now(UTC).date(), freq="M") - 1
)
flows_month = fetch_flows_range(
    month.start_time.strftime("%Y-%m-%d"), (month + 1).start_time.strftime("%Y-%m-%d")
)
leaderboard_month = flow_type_summary(flows_month, "Congestion")
if exclude_ukraine:
    leaderboard_month = drop_ukraine(leaderboard_month)

fig = leaderboard_subplots(leaderboard_month, f"Top 10 borders by congestion metric, {month}")
save_chart(fig, f"monthly-leaderboard-{month}")
fig.show()

### Yearly leaderboard

Same four metrics, rolled up over a full calendar year instead of a month &ndash; useful for spotting borders that are a persistent, structural bottleneck.

In [ ]:
selected_year = None  # e.g. 2023; None = last full calendar year

year = selected_year or (datetime.now(UTC).year - 1)
# first run re-fetches ~365 daily chunks, cached afterwards
flows_year = fetch_flows_range(f"{year}-01-01", f"{year + 1}-01-01")
leaderboard_year = flow_type_summary(flows_year, "Congestion")
if exclude_ukraine:
    leaderboard_year = drop_ukraine(leaderboard_year)

fig = leaderboard_subplots(leaderboard_year, f"Top 10 borders by congestion metric, {year}")
save_chart(fig, f"yearly-leaderboard-{year}")
fig.show()

## Concentration check: how much of the rent do the top borders capture?

The leaderboards above show a steep drop-off, which is the usual basis for claiming a few borders account for most of the congestion rent. This measures that instead of asserting it.

The pies rank by `implied_value_eur` (congestion rent, EUR) specifically &ndash; not `n_hours` or `volume_mwh`, which would give a different ordering &ndash; and split each period's total rent into the top 10 borders by that metric vs. everyone else. Both inherit the `exclude_ukraine` setting from above.

In [ ]:
month_split = top_vs_rest(leaderboard_month)
year_split = top_vs_rest(leaderboard_year)

month_top_share = 1 - month_split["implied_value_eur"].iloc[-1] / month_split["implied_value_eur"].sum()
year_top_share = 1 - year_split["implied_value_eur"].iloc[-1] / year_split["implied_value_eur"].sum()
print(
    f"Top 10 borders capture {month_top_share:.0%} of {month}'s rent "
    f"(of {len(leaderboard_month)} borders), "
    f"{year_top_share:.0%} of {year}'s (of {len(leaderboard_year)})."
)

fig = concentration_pie_chart(
    month_split,
    year_split,
    f"{month} (congestion rent, EUR)",
    f"{year} (congestion rent, EUR)",
)
save_chart(fig, f"rent-concentration-{month}-vs-{year}")
fig.show()

## Congestion rent by region and country

Grouping borders by region gives a coarser view than the per-border leaderboard: is congestion rent concentrated within one part of Europe's grid, or spread evenly? Regions here are an approximate, geography-based grouping loosely modeled on ENTSO-E's [Capacity Calculation Regions](https://www.entsoe.eu/network_codes/ccr-regions/) (Nordic, Baltic, Core, Italy North, Greece-Italy, South-East Europe, South-West Europe), coloured to match ENTSO-E's public CCR map. It's a simplification, not the real (overlapping) CCR membership: several TSOs legally sit in more than one CCR, which a one-region-per-zone mapping can't represent, so any border whose two sides fall in different regions is bucketed as "Cross-region" rather than forced into either side.

The treemap below drills region &rarr; country &rarr; border, sized and coloured by `implied_value_eur` (congestion rent, EUR) for the same year as the yearly leaderboard above. The country level only appears for countries split across multiple bidding zones (Norway, Sweden, Denmark, Italy) &ndash; for everywhere else, a country node would just relabel a single border, so those borders attach straight to their region instead.

In [ ]:
fig = region_treemap(leaderboard_year, year)
save_chart(fig, f"rent-by-region-country-{year}")
fig.show()

## Congestion rent by real CCR mechanism

The treemap above groups borders by loose geography. Europe's actual capacity-allocation rules are grouped by mechanism instead &ndash; explicit auction, price-coupled NTC (radial or lineset), or price-coupled flow-based &ndash; and don't respect geography: a single country's borders can fall under different mechanisms depending which neighbor is on the other side. The breakdown below follows Iacopo Bertelli's per-border mapping from [Counterintuitive Flows in 2023, Part 3](https://www.linkedin.com/pulse/counterintuitive-flows-2023-part-3-iacopo-bertelli-xjetf/), with Italy North's borders folded into Core &ndash; ACER decisions 04/2024 and 10/2025 merge the two CCRs (officially renaming the result "Central Europe", though "Core" is still the term in common use; day-ahead go-live for the merged region isn't until 2027).

This is per-border, not per-zone, since real CCR membership overlaps (Germany-Luxembourg sits in both Core and Hansa, France in both Core and SWE) in a way no single-region-per-zone map can represent. Borders outside Iacopo's breakdown &ndash; Western Balkans, Ukraine, Sardinia &ndash; fall back to "Other".

<!-- TODO: trim research detail -->

In [ ]:
fig = ccr_mechanism_treemap(leaderboard_year, year)
save_chart(fig, f"rent-by-ccr-mechanism-{year}")
fig.show()

## Takeaways


In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(
        f"- The top 10 borders capture {month_top_share:.0%} of {month}'s congestion rent "
        f"({len(leaderboard_month)} borders in total) and {year_top_share:.0%} of {year}'s "
        f"({len(leaderboard_year)})."
    )
)

- Which border is "worst" depends on the metric. Hours flagged, volume, total rent, and rent-per-hour produce visibly different top 10s in the leaderboards above, so any single ranking is a choice about the question being asked.
- Try narrowing `fetch_flows`/`fetch_flows_range` to a single border, or widening the window past 30 days/one month, to see how a specific bottleneck evolves.

---

Data: [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai).

# Appendix

## The leaderboard: highest-rent borders

A handful of interconnectors usually account for the bulk of the congestion rent. That's a claim worth testing rather than asserting, so the concentration check further down measures it directly.


In [ ]:
leaderboard_30d = flow_type_summary(flows_30d, "Congestion")
if exclude_ukraine:
    leaderboard_30d = drop_ukraine(leaderboard_30d)

fig = top_n_bar(
    leaderboard_30d,
    "implied_value_eur",
    "Congestion rent (EUR)",
    "Top 15 borders by congestion rent (last 30 days)",
    FLOW_COLORS["Congestion"],
    top_n=15,
)
save_chart(fig, "top15-borders-congestion-rent-last-30-days")
fig.show()

## Congestion rent over time

Daily totals across all borders show whether rent is trending, seasonal, or driven by a handful of spike days.

In [ ]:
fig = daily_rent_chart(congested)
save_chart(fig, "daily-congestion-rent-last-30-days")
fig.show()